# Step 3 Entity-Mention DP Review

Manual walkthrough of `align_entity_mentions` with concrete examples.
Goal: verify the mechanism and understand why it measured negative (0.23 vs 0.54 baseline).

In [2]:
import sys
sys.path.insert(0, '/Users/rikhoekstra/develop/republic_ner_matching')
from scripts.s6b_known_point_ledger import align_entity_mentions
import pandas as pd
import numpy as np

## The DP Moves

Three unit-cost moves from cell `(i, j)`:
1. **Match** `(i+1, j+1)`: cost 0 if flat[i]==enriched[j], else 1
2. **Flat-skip** `(i+1, j)`: cost 1 (HTR side unmatched, continue enriched)
3. **Enriched-run** `(i, j+1)`: cost 1 (enriched mentions accumulate, stay on same HTR position)

The third move is the fix 2026-09-28: enables "many-to-one *in both* directions."

## Example 1: Flat-skip (original work, n > m case)

In [3]:
flat =    ["Holland",  "Zeeland", "Utrecht"]
enriched = ["Holland",             "Utrecht"]

pairs, cost = align_entity_mentions(flat, enriched)
print(f"Flat:     {flat}")
print(f"Enriched: {enriched}")
print(f"Pairs: {pairs}")
print(f"Cost: {cost}")
print()
print("Path: (0,0) --match--> (1,1) --flat-skip--> (2,1) --match--> (3,2)")
print("      Holland matches, Zeeland is skipped (HTR-internal detail), Utrecht matches")

Flat:     ['Holland', 'Zeeland', 'Utrecht']
Enriched: ['Holland', 'Utrecht']
Pairs: [(0, 0), (2, 1)]
Cost: 1.0

Path: (0,0) --match--> (1,1) --flat-skip--> (2,1) --match--> (3,2)
      Holland matches, Zeeland is skipped (HTR-internal detail), Utrecht matches


## Example 2: Enriched-run (new work, n < m case) 

In [4]:
flat =    ["Holland",  "Zeeland"]
enriched = ["Holland", "Vriesland", "Zeeland"]

pairs, cost = align_entity_mentions(flat, enriched)
print(f"Flat:     {flat}")
print(f"Enriched: {enriched}")
print(f"Pairs: {pairs}")
print(f"Cost: {cost}")
print()
print("Path: (0,0) --match--> (1,1) --enriched-run--> (1,2) --match--> (2,3)")
print("      Holland matches (i+1,j+1), then Vriesland,Zeeland are enriched-run on same HTR,")
print("      then final match at position (1,2).")
print()
print("BEFORE the fix, this case was unreachable (n=2 < m=3): DP returned [], cost=inf")
print("This is the corpus's dominant shape (81.1% HTR under-segmentation): many enriched")
print("resolutions map to one HTR unit. The fix enables expressing it.")

Flat:     ['Holland', 'Zeeland']
Enriched: ['Holland', 'Vriesland', 'Zeeland']
Pairs: [(0, 0), (1, 2)]
Cost: 1.0

Path: (0,0) --match--> (1,1) --enriched-run--> (1,2) --match--> (2,3)
      Holland matches (i+1,j+1), then Vriesland,Zeeland are enriched-run on same HTR,
      then final match at position (1,2).

BEFORE the fix, this case was unreachable (n=2 < m=3): DP returned [], cost=inf
This is the corpus's dominant shape (81.1% HTR under-segmentation): many enriched
resolutions map to one HTR unit. The fix enables expressing it.


## Example 3: Detailed DP table — why the new path fails on real data

In [5]:
# Simulated real data: lots of vocabulary hits, but they don't align well
flat = ["amsterdam", "rotterdam", "leiden", "amsterdam", "den", "haag"]  # 6 HTR mentions
enriched = ["leiden", "amsterdam", "rotterdam"]  # 3 enriched mentions, different order

pairs, cost = align_entity_mentions(flat, enriched)
print(f"Flat (n={len(flat)}):     {flat}")
print(f"Enriched (m={len(enriched)}): {enriched}")
print(f"Pairs: {pairs}")
print(f"Cost: {cost}")
print()
print("Why this fails on the real corpus:")
print("1. Order is violated: enriched=[leiden, amsterdam, rotterdam]")
print("   but flat has [amsterdam, rotterdam, leiden, amsterdam, ...]")
print("2. No monotone path exists that respects BOTH sequences AND the DP constraints.")
print("3. The aligner can only match when the sequences happen to align.")
print()
print("On the 52 open-gap resolutions, 27 of 48 projected pairs had ZERO shared vocabulary,")
print("and even the 21 with identical names landed mostly by positional accident, not evidence.")

Flat (n=6):     ['amsterdam', 'rotterdam', 'leiden', 'amsterdam', 'den', 'haag']
Enriched (m=3): ['leiden', 'amsterdam', 'rotterdam']
Pairs: [(2, 0), (3, 1), (5, 2)]
Cost: 4.0

Why this fails on the real corpus:
1. Order is violated: enriched=[leiden, amsterdam, rotterdam]
   but flat has [amsterdam, rotterdam, leiden, amsterdam, ...]
2. No monotone path exists that respects BOTH sequences AND the DP constraints.
3. The aligner can only match when the sequences happen to align.

On the 52 open-gap resolutions, 27 of 48 projected pairs had ZERO shared vocabulary,
and even the 21 with identical names landed mostly by positional accident, not evidence.


## Summary: Why Step 3 Measured Negative

1. **Mechanism works correctly**: the three moves execute as specified, including the new enriched-run.

2. **Coverage improved**: word boundaries removed junk vocabulary, allowing 45/52 open-gap resolutions 
   to be projected (vs 20 before). That's real progress on one metric.

3. **But accuracy collapsed**: exact-start agreement 0.23 vs baseline 0.54. Why?
   - Junk vocabulary (`over`, `inde`, etc.) had been propping up the HTR-side mention count `n`.
   - Removing it made many gaps suddenly reachable.
   - But those gaps align with POSITIONS, not entity evidence.
   - 73% of the new projections have zero shared vocabulary and place resolutions wrong by mean 6.9 paragraphs.

4. **The fundamental issue**: even with the fix, the DP operates on symbol-order alone. 
   When two text streams don't align naturally (different order, different mention sets, different density),
   the aligner falls back to positional matching — which beats the pure even-split baseline ONLY if the
   streams happen to be roughly co-ordered. On real enriched vs HTR, they are not.

The conclusion: **the mechanism is not the problem; the evidence is.** Word-boundary filtering was a correct
fix to the vocabulary, but it exposed the deeper limit: entity-mention order alone cannot align streams
with fundamentally different segmentation, density, and mention sets.